In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectPercentile
from sklearn.feature_selection import chi2
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

from scipy.sparse import hstack
import nltk

from subprocess import check_output
#print(check_output(["ls", "../input"]).decode("utf8"))


In [2]:
df_train = pd.read_csv('../input/train.csv')
df_predict = pd.read_csv('../input/test.csv')

all_text = pd.concat([df_train['comment_text'], df_predict['comment_text']])


In [3]:
# Adding new features with RegEx

def add_features(df):
    # new column for exclamation mark
    df['ex_mark'] = df['comment_text'].str.findall('\!')
    df['ex_mark'] = df['ex_mark'].apply(lambda x: len(x))
    df['ex_mark'] = df['ex_mark'].apply(lambda x: (1 if x >= 1 else 0))
    
    # new column for question mark
    df['qu_mark'] = df['comment_text'].str.findall('\?')
    df['qu_mark'] = df['qu_mark'].apply(lambda x: len(x))
    df['qu_mark'] = df['qu_mark'].apply(lambda x: (1 if x >= 1 else 0))

    # new columns for smileys
    smileys_good = '((:|;)-?(\)|P|D))'
    smileys_bad = '((:|;)-?\'?(\())'
    df['smileys_good'] = df['comment_text'].str.extract(smileys_good, expand=True)[0].fillna(0)
    df['smileys_bad'] = df['comment_text'].str.extract(smileys_bad, expand=True)[0].fillna(0)

    df['smileys_good'][df['smileys_good']!=0] = 1
    df['smileys_bad'][df['smileys_bad']!=0] = 1
    
    return df
    
df_train = add_features(df_train)
df_predict = add_features(df_predict)

df_train.head()


/tmp/ipykernel_11/1778889289.py:20: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['smileys_good'][df['smileys_good']!=0] = 1
/tmp/ipykernel_11/1778889289.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice fro

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,ex_mark,qu_mark,smileys_good,smileys_bad
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,0,1,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,1,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,0,1,0,0


In [4]:
# nltk data preprocessing here
'''Lemmatizer did not improve the cv-score'''

# Tokenization + Stemming
#def clean_data(df):
#    Lemmatizer = nltk.WordNetLemmatizer()
#    def stem_text(text):
#        return [Lemmatizer.lemmatize(w) for w in nltk.word_tokenize(text)]
#
#    df['comment_text'] = df['comment_text'].apply(stem_text)
#
#    # array back to string
#    df['comment_text'] = df['comment_text'].apply(lambda x: ' '.join(x))
#    return df
    
#df_train = clean_data(df_train)
#df_predict = clean_data(df_predict)


'Lemmatizer did not improve the cv-score'

In [5]:
# Fit the CountVectorizer to all data
vect = CountVectorizer(min_df=4, ngram_range=(1,3), stop_words='english').fit(all_text)


In [6]:
# Split data into X and Y
X_train = df_train[['comment_text', 'smileys_good', 'smileys_bad', 'ex_mark', 'qu_mark']]
X_predict = df_predict[['comment_text', 'smileys_good', 'smileys_bad', 'ex_mark', 'qu_mark']]
Y = df_train[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']]

# transform the documents in the training data to a document-term matrix
X_train_vectorized = vect.transform(X_train['comment_text'])
X_predict_vectorized = vect.transform(X_predict['comment_text'])


In [7]:
#print p-scores
scores = chi2(X_train_vectorized, Y)[1]
chis = pd.DataFrame(list(zip(vect.get_feature_names(), scores))).sort_values(by=1)
print('Perc. of rel. features: ' + str(chis[1][chis[1]<0.1].count()/chis[1].count()))

# plot influence of dimensionality reduction
#model = Pipeline([('chi2', SelectPercentile(chi2)), ('svc', LinearSVC())])
#params = {'C': [1], 'random_state' : [0]}
#percentiles = (15, 35, 65, 100)

#for y_col in Y.columns:
#    score_means = []
#    score_stds = []
#    for percentile in percentiles:
#        model.set_params(chi2__percentile=percentile)
#        # Compute cross-validation score using 1 CPU
#        this_scores = cross_val_score(model, train_features, Y[y_col], n_jobs=1)
#        score_means.append(this_scores.mean())
#
#    plt.errorbar(percentiles, score_means, np.array(score_stds))
#    plt.title(y_col)
#    plt.xlabel('Percentile')
#    plt.ylabel('Prediction rate')
#    plt.show()


# Chi²-Filter
chi2_filter = SelectPercentile(chi2, 0.25)
X_train_filtered = chi2_filter.fit_transform(X_train_vectorized, Y)
X_predict_filtered = chi2_filter.transform(X_predict_vectorized)


# add features
train_features = hstack([X_train_filtered, 
                         X_train[['smileys_good', 'smileys_bad', 'ex_mark', 'qu_mark']].astype('int64')])

predict_features = hstack([X_predict_filtered, 
                         X_predict[['smileys_good', 'smileys_bad', 'ex_mark', 'qu_mark']].astype('int64')])


/usr/local/lib/python3.11/dist-packages/sklearn/feature_selection/_univariate_selection.py:165: RuntimeWarning: divide by zero encountered in divide
  chisq /= f_exp


AttributeError: 'CountVectorizer' object has no attribute 'get_feature_names'

In [8]:
#Tune hyperparameters with GridsearchCV

model = LinearSVC()
params = {'C': [1], 'random_state' : [0]}

Y_predicted = pd.DataFrame()
Y_predicted['id'] = df_predict['id']
scores = []

for y_col in Y.columns:
    gsCV = GridSearchCV(model, params, scoring="roc_auc").fit(train_features, Y[y_col])
    scoreX = np.mean(gsCV.cv_results_['mean_test_score'])
    scores.append(scoreX)
    Y_predicted[y_col] = gsCV.predict(predict_features)
    print(y_col + ':' + str(scoreX))

print('mean score: ' + str(np.mean(scores)))    


NameError: name 'train_features' is not defined

In [9]:
submission = Y_predicted
submission.to_csv('submission.csv', index=False)
